# เทรนโมเดลตรวจจับวัตถุบนสายพานด้วย Google Colab

Notebook สำหรับ Lab 10 ใช้เทรน YOLO Object Detection จากชุดข้อมูลที่เตรียมไว้บนคอมพิวเตอร์ ก่อนเริ่มเทรน ให้เลือก **Runtime > Change runtime type > T4 GPU** (หรือ GPU รุ่นอื่น) ใน Colab

## ทำ Bounding Box ใน Colab ก่อนเทรน

GUI เก็บภาพยังไม่ได้สร้าง Bounding Box: Notebook นี้ให้โมเดล YOLO-World เสนอกรอบ แล้วให้คุณ **ตรวจและแก้ไขทุกภาพ** ก่อนสร้างชุดข้อมูล YOLO และเทรน กรอบอัตโนมัติอาจผิดหรือหาวัตถุไม่เจอ โดยเฉพาะวัตถุเฉพาะทางและความแตกต่างที่มองเห็นยาก

นำโฟลเดอร์คลาสที่ได้จาก GUI ไปบีบอัดเป็น `conveyor_images.zip` แล้วอัปโหลดไปที่ My Drive โครงสร้างภายใน ZIP ต้องเป็น `captured_images/ชื่อคลาส/ภาพ.jpg` หรือ `ชื่อคลาส/ภาพ.jpg` หากมีชุดข้อมูล YOLO ที่ทำ label แล้ว สามารถเลือกโหมด `prepared` ในเซลล์ตั้งค่าแทนได้ ควรลบภาพซ้ำหรือเกือบซ้ำก่อนแบ่งชุด เพื่อไม่ให้คะแนน Validation ดูดีเกินจริง

In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics", "pyyaml", "jupyter-bbox-widget==0.7.0", "ipywidgets", "git+https://github.com/ultralytics/CLIP.git"])

## เชื่อมต่อ Google Drive และเลือกไฟล์ชุดข้อมูล ZIP

เลือก `raw` สำหรับภาพจาก GUI หรือ `prepared` สำหรับชุดข้อมูล YOLO ที่ทำ label และแบ่ง Train/Val แล้ว จากนั้นแก้ path ให้ตรงกับไฟล์ใน **My Drive** หากชื่อโฟลเดอร์คลาสเป็นภาษาไทยหรือเป็นชื่อเฉพาะ ให้ระบุคำอธิบายวัตถุเป็นภาษาอังกฤษใน `PROMPT_OVERRIDES` เช่น `{"ขวดดี": "bottle"}` เพื่อช่วยโมเดลเสนอกรอบ (ยังต้องตรวจคลาสเอง)

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")
MODE = "raw"  # "raw" หรือ "prepared"
RAW_ZIP = Path("/content/drive/MyDrive/conveyor_images.zip")
DATASET_ZIP = Path("/content/drive/MyDrive/conveyor_dataset.zip")
PROMPT_OVERRIDES = {}  # เช่น {"ชื่อโฟลเดอร์": "English object description"}
AUTO_CONFIDENCE = 0.15
if MODE not in {"raw", "prepared"}:
    raise ValueError("MODE must be 'raw' or 'prepared'")
input_zip = RAW_ZIP if MODE == "raw" else DATASET_ZIP
if not input_zip.is_file():
    raise FileNotFoundError(f"ZIP not found: {input_zip}")
print(f"Input ZIP: {input_zip}")

## เสนอ Bounding Box อัตโนมัติและตรวจแก้ใน Colab (โหมด raw)

รันเซลล์ถัดไปเพื่อเตรียมภาพและโหลดโมเดล จากนั้นรันเซลล์ตรวจภาพทีละภาพ โมเดลจะเสนอกรอบตามคลาสของโฟลเดอร์ภาพนั้น จึงต้องเพิ่มกรอบของวัตถุคลาสอื่นที่มีในภาพเอง เลือกคลาสแล้วลากเพิ่มกรอบ หรือเลือกกรอบเพื่อแก้/ลบ กด **Submit** เมื่อตรวจภาพนั้นครบแล้ว; **Skip** ข้ามโดยยังไม่ถือว่าตรวจ หากภาพไม่มีวัตถุจริง ให้เลือกช่องยืนยันภาพว่างก่อนกด Submit ผลตรวจจะบันทึกใน Drive และกลับมาทำต่อได้เมื่อใช้ ZIP และการตั้งค่าเดิม

In [ ]:
if MODE == "raw":
    import hashlib
    import json
    import tempfile
    import zipfile
    from pathlib import PurePosixPath
    import torch
    from ultralytics import YOLO

    if not torch.cuda.is_available():
        raise RuntimeError("ไม่พบ GPU: เลือก Runtime > Change runtime type > GPU แล้วรันใหม่")
    raw_dir = Path(tempfile.mkdtemp(prefix="conveyor_raw_"))
    with zipfile.ZipFile(RAW_ZIP) as archive:
        for member in archive.infolist():
            relative = PurePosixPath(member.filename.replace("\\", "/"))
            if relative.is_absolute() or any(part == ".." or ":" in part for part in relative.parts):
                raise ValueError(f"Unsafe ZIP entry: {member.filename}")
            target = raw_dir.joinpath(*relative.parts)
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(member) as source, target.open("wb") as destination:
                    import shutil
                    shutil.copyfileobj(source, destination)
    raw_images = sorted(p for p in raw_dir.rglob("*") if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"} and "__MACOSX" not in p.parts)
    if len(raw_images) < 2 or any(p.parent == raw_dir for p in raw_images):
        raise ValueError("ต้องมีภาพอย่างน้อย 2 ภาพ และแต่ละภาพต้องอยู่ในโฟลเดอร์คลาส")
    class_names = sorted({p.parent.name for p in raw_images})
    unknown = set(PROMPT_OVERRIDES) - set(class_names)
    if unknown:
        raise ValueError(f"ไม่พบโฟลเดอร์คลาสที่ระบุใน PROMPT_OVERRIDES: {sorted(unknown)}")
    prompts = {name: PROMPT_OVERRIDES.get(name, name) for name in class_names}
    if any(not isinstance(value, str) or not value.strip() for value in prompts.values()):
        raise ValueError("คำอธิบายคลาสต้องเป็นข้อความที่ไม่ว่าง")
    if not 0 < AUTO_CONFIDENCE < 1:
        raise ValueError("AUTO_CONFIDENCE ต้องอยู่ระหว่าง 0 และ 1")
    digest = hashlib.sha256()
    with RAW_ZIP.open("rb") as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(chunk)
    digest.update(json.dumps([prompts, AUTO_CONFIDENCE], ensure_ascii=False, sort_keys=True).encode("utf-8"))
    review_dir = Path("/content/drive/MyDrive/conveyor_reviews") / digest.hexdigest()
    review_dir.mkdir(parents=True, exist_ok=True)
    def review_path(image):
        key = hashlib.sha256(image.relative_to(raw_dir).as_posix().encode("utf-8")).hexdigest()
        return review_dir / f"{key}.json"
    detector = YOLO("yolov8s-worldv2.pt")
    print(f"ภาพ: {len(raw_images)} | คลาส: {class_names}")
    print(f"ตำแหน่งบันทึกผลตรวจ: {review_dir}")
else:
    print("ใช้ชุดข้อมูลที่เตรียมไว้แล้ว: ข้ามขั้นตอนเสนอกรอบ")

In [ ]:
if MODE == "raw":
    import html
    import ipywidgets as widgets
    from google.colab import output
    from IPython.display import display
    from jupyter_bbox_widget import BBoxWidget
    from PIL import Image

    output.enable_custom_widget_manager()
    box_widget = BBoxWidget(classes=class_names)
    status = widgets.HTML()
    empty_confirm = widgets.Checkbox(value=False, description="ยืนยันว่าภาพนี้ไม่มีวัตถุ")
    previous_button = widgets.Button(description="ภาพก่อนหน้า")
    messages = widgets.Output()
    state = {"index": 0, "prompt": None}

    def show_review(index):
        state["index"] = index % len(raw_images)
        image = raw_images[state["index"]]
        empty_confirm.value = False
        box_widget.image = str(image)
        box_widget.label = image.parent.name
        box_widget.bboxes = []
        reviewed = sum(review_path(p).is_file() for p in raw_images)
        status.value = f"ภาพ {state['index'] + 1}/{len(raw_images)}: {html.escape(image.relative_to(raw_dir).as_posix())} | ตรวจแล้ว {reviewed}/{len(raw_images)}"
        saved = review_path(image)
        if saved.is_file():
            box_widget.bboxes = json.loads(saved.read_text(encoding="utf-8"))["boxes"]
        else:
            prompt = prompts[image.parent.name]
            if state["prompt"] != prompt:
                detector.set_classes([prompt])
                state["prompt"] = prompt
            result = detector.predict(source=str(image), conf=AUTO_CONFIDENCE, device=0, verbose=False)[0]
            box_widget.bboxes = [
                {"x": float(x1), "y": float(y1), "width": float(x2 - x1), "height": float(y2 - y1), "label": image.parent.name}
                for x1, y1, x2, y2 in result.boxes.xyxy.cpu().tolist()
            ]

    @messages.capture(clear_output=True)
    def save_review():
        image = raw_images[state["index"]]
        boxes = [dict(box) for box in box_widget.bboxes]
        if not boxes and not empty_confirm.value:
            raise ValueError("หากภาพไม่มีวัตถุจริง ให้เลือกช่องยืนยันภาพว่างก่อนบันทึก")
        with Image.open(image) as opened:
            width, height = opened.size
        for box in boxes:
            x, y, w, h = (float(box[key]) for key in ("x", "y", "width", "height"))
            if box.get("label") not in class_names or not (0 <= x < width and 0 <= y < height and w > 0 and h > 0 and x + w <= width + 0.01 and y + h <= height + 0.01):
                raise ValueError(f"กรอบหรือคลาสไม่ถูกต้อง: {box}")
        target = review_path(image)
        temporary = target.with_suffix(".tmp")
        temporary.write_text(json.dumps({"boxes": boxes}, ensure_ascii=False), encoding="utf-8")
        temporary.replace(target)
        show_review(state["index"] + 1)

    @messages.capture(clear_output=True)
    def skip_review():
        show_review(state["index"] + 1)

    @messages.capture(clear_output=True)
    def previous_review(_):
        show_review(state["index"] - 1)

    box_widget.on_submit(save_review)
    box_widget.on_skip(skip_review)
    previous_button.on_click(previous_review)
    first_unreviewed = next((i for i, image in enumerate(raw_images) if not review_path(image).is_file()), 0)
    show_review(first_unreviewed)
    display(widgets.VBox([status, empty_confirm, previous_button, box_widget, messages]))
else:
    print("ใช้ชุดข้อมูลที่เตรียมไว้แล้ว: ข้ามการตรวจกรอบ")

## สร้างชุดข้อมูลจากกรอบที่ตรวจแล้ว

หลังจากกด Submit ครบทุกภาพ ให้รันเซลล์นี้เพื่อแบ่ง Train/Validation และสร้างไฟล์ YOLO อัตโนมัติ ภาพที่กด Skip ไว้ยังไม่ถือว่าตรวจ และจะเทรนต่อไม่ได้

In [ ]:
if MODE == "raw":
    import random
    import shutil
    import yaml
    from PIL import Image

    missing_reviews = [p for p in raw_images if not review_path(p).is_file()]
    if missing_reviews:
        raise ValueError(f"ยังไม่ได้ตรวจ {len(missing_reviews)} ภาพ: กลับไปตรวจและกด Submit ให้ครบ")
    shuffled = list(raw_images)
    random.Random(42).shuffle(shuffled)
    val_count = max(1, int(len(shuffled) * 0.2))
    positive = [image for image in shuffled if json.loads(review_path(image).read_text(encoding="utf-8"))["boxes"]]
    if len(positive) < 2:
        raise ValueError("ต้องมีภาพที่มีกรอบวัตถุอย่างน้อย 2 ภาพสำหรับแบ่ง Train และ Validation")
    val_images = {positive[0]}
    for image in shuffled:
        if len(val_images) >= val_count:
            break
        if image != positive[1]:
            val_images.add(image)
    build_dir = Path(tempfile.mkdtemp(prefix="conveyor_yolo_"))
    for split in ("train", "val"):
        (build_dir / "images" / split).mkdir(parents=True)
        (build_dir / "labels" / split).mkdir(parents=True)
    split_box_counts = {"train": 0, "val": 0}
    class_counts = {split: {name: 0 for name in class_names} for split in ("train", "val")}
    for image in shuffled:
        split = "val" if image in val_images else "train"
        image_id = review_path(image).stem
        shutil.copy2(image, build_dir / "images" / split / f"{image_id}{image.suffix.lower()}")
        boxes = json.loads(review_path(image).read_text(encoding="utf-8"))["boxes"]
        with Image.open(image) as opened:
            width, height = opened.size
        lines = []
        for box in boxes:
            x, y, w, h = (float(box[key]) for key in ("x", "y", "width", "height"))
            class_id = class_names.index(box["label"])
            class_counts[split][box["label"]] += 1
            lines.append(f"{class_id} {(x + w / 2) / width:.8f} {(y + h / 2) / height:.8f} {w / width:.8f} {h / height:.8f}")
        split_box_counts[split] += len(lines)
        (build_dir / "labels" / split / f"{image_id}.txt").write_text("\n".join(lines) + ("\n" if lines else ""), encoding="utf-8")
    if any(count == 0 for count in split_box_counts.values()):
        raise ValueError(f"Train และ Validation ต้องมีกรอบวัตถุอย่างน้อยชุดละ 1 กรอบ: {split_box_counts}")
    (build_dir / "data.yaml").write_text(yaml.safe_dump({"train": "images/train", "val": "images/val", "names": class_names}, allow_unicode=True, sort_keys=False), encoding="utf-8")
    DATASET_ZIP = build_dir.parent / f"{build_dir.name}.zip"
    with zipfile.ZipFile(DATASET_ZIP, "w", zipfile.ZIP_DEFLATED) as archive:
        for path in build_dir.rglob("*"):
            if path.is_file():
                archive.write(path, path.relative_to(build_dir).as_posix())
    print(f"สร้างชุดข้อมูลแล้ว: {DATASET_ZIP} | จำนวนกรอบ: {split_box_counts}")
    print(f"จำนวนกรอบแยกตามคลาส: {class_counts}")
    if any(count == 0 for counts in class_counts.values() for count in counts.values()):
        print("WARNING: มีคลาสที่ไม่มีกรอบใน Train หรือ Validation; เพิ่มภาพที่หลากหลายและตรวจการแบ่งชุดข้อมูล")
else:
    print(f"ใช้ชุดข้อมูลที่เตรียมไว้แล้ว: {DATASET_ZIP}")

## แตกไฟล์และตรวจสอบชุดข้อมูล YOLO

เซลล์ถัดไปตรวจว่าแต่ละภาพในชุด Train/Validation มีไฟล์ label YOLO ที่ชื่อคู่กัน พร้อมตรวจ class ID และพิกัดกรอบว่าอยู่ในรูปแบบที่ถูกต้อง

In [ ]:
import math
import shutil
import tempfile
import zipfile
from pathlib import PurePosixPath

import yaml

IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
EXTRACT_DIR = Path(tempfile.mkdtemp(prefix="conveyor_dataset_"))

with zipfile.ZipFile(DATASET_ZIP) as archive:
    for member in archive.infolist():
        relative = PurePosixPath(member.filename.replace("\\", "/"))
        if relative.is_absolute() or any(part in {"..", ""} or ":" in part for part in relative.parts):
            raise ValueError(f"Unsafe ZIP entry: {member.filename}")
        target = EXTRACT_DIR.joinpath(*relative.parts)
        if member.is_dir():
            target.mkdir(parents=True, exist_ok=True)
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as source, target.open("wb") as destination:
                shutil.copyfileobj(source, destination)

yaml_files = list(EXTRACT_DIR.rglob("data.yaml"))
if len(yaml_files) != 1:
    raise ValueError(f"Expected exactly one data.yaml in the ZIP; found {len(yaml_files)}")
DATA_YAML = yaml_files[0]
DATASET_DIR = DATA_YAML.parent
dataset_config = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))

names = dataset_config.get("names")
if isinstance(names, dict):
    names = [str(names[key]) for key in sorted(names, key=lambda value: int(value))]
elif isinstance(names, list):
    names = [str(name) for name in names]
else:
    raise ValueError("data.yaml must define class names as a list or integer-keyed mapping")
if not names or len(set(names)) != len(names):
    raise ValueError("Class names must be non-empty and unique")
if int(dataset_config.get("nc", len(names))) != len(names):
    raise ValueError(f"nc does not match the number of names: nc={dataset_config.get('nc')}, names={len(names)}")

dataset_config["path"] = str(DATASET_DIR)
box_counts = {name: 0 for name in names}
summary = {}
for split in ("train", "val"):
    image_dir = DATASET_DIR / "images" / split
    label_dir = DATASET_DIR / "labels" / split
    if not image_dir.is_dir() or not label_dir.is_dir():
        raise FileNotFoundError(f"Missing images/{split} or labels/{split} under {DATASET_DIR}")
    images = sorted(path for path in image_dir.rglob("*") if path.suffix.lower() in IMAGE_SUFFIXES)
    if not images:
        raise ValueError(f"No images found in {image_dir}")
    empty_labels = 0
    for image_path in images:
        label_path = label_dir / image_path.relative_to(image_dir).with_suffix(".txt")
        if not label_path.is_file():
            raise FileNotFoundError(f"Missing label for {image_path}: expected {label_path}")
        lines = [line.split() for line in label_path.read_text(encoding="utf-8-sig").splitlines() if line.strip()]
        if not lines:
            empty_labels += 1
        for line_number, fields in enumerate(lines, start=1):
            if len(fields) != 5:
                raise ValueError(f"{label_path}:{line_number}: expected class_id x_center y_center width height")
            class_id = int(fields[0])
            values = [float(value) for value in fields[1:]]
            if class_id < 0 or class_id >= len(names):
                raise ValueError(f"{label_path}:{line_number}: class id {class_id} is outside 0..{len(names) - 1}")
            if not all(math.isfinite(value) and 0 <= value <= 1 for value in values):
                raise ValueError(f"{label_path}:{line_number}: box values must be finite and normalized to 0..1")
            if values[2] == 0 or values[3] == 0:
                raise ValueError(f"{label_path}:{line_number}: box width and height must be greater than zero")
            box_counts[names[class_id]] += 1
    summary[split] = {"images": len(images), "empty_labels": empty_labels}

if sum(box_counts.values()) == 0:
    raise ValueError("No bounding boxes were found in the dataset labels")
for split in ("train", "val"):
    dataset_config[split] = f"images/{split}"
DATA_YAML.write_text(yaml.safe_dump(dataset_config, sort_keys=False, allow_unicode=True), encoding="utf-8")

print(f"Dataset: {DATASET_DIR}")
print(f"Classes: {names}")
print(f"Splits: {summary}")
print(f"Bounding boxes by class: {box_counts}")
print(f"Validated data.yaml: {DATA_YAML}")

## เทรนและประเมินผล

ค่าเริ่มต้นตามสคริปต์เทรนในโปรเจกต์คือ `yolo11n.pt`, 50 epochs, ขนาดภาพ 640 พิกเซล และ batch size 8 ควรปรับค่าเมื่อเตรียมชุดข้อมูลพร้อมแล้วเท่านั้น หาก Colab ไม่พบ GPU เซลล์จะหยุดและแจ้งให้เปลี่ยน Runtime เป็น GPU

In [ ]:
from datetime import datetime

import torch
from ultralytics import YOLO

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. In Colab, choose Runtime > Change runtime type > GPU, then rerun.")
print(f"GPU: {torch.cuda.get_device_name(0)}")

MODEL = "yolo11n.pt"
EPOCHS = 50
IMAGE_SIZE = 640
BATCH_SIZE = 8
RUN_NAME = datetime.now().strftime("conveyor_%Y%m%d_%H%M%S")
RUN_DIR = Path("/content/conveyor_runs/detect") / RUN_NAME

model = YOLO(MODEL)
model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=0,
    project=str(RUN_DIR.parent),
    name=RUN_DIR.name,
)

BEST_PT = RUN_DIR / "weights" / "best.pt"
if not BEST_PT.is_file():
    raise FileNotFoundError(f"Training finished without the expected model: {BEST_PT}")
print(f"Best model: {BEST_PT}")

In [ ]:
best_model = YOLO(str(BEST_PT))
metrics = best_model.val(data=str(DATA_YAML), device=0)
print("Validation metrics:", metrics.results_dict)

## บันทึกโมเดลกลับไปยัง Google Drive

เก็บผล Validation ไว้กับโมเดล เพื่อใช้เปรียบเทียบผลการเทรนแต่ละรอบภายหลัง

In [ ]:
from google.colab import files

DRIVE_OUTPUT = Path("/content/drive/MyDrive/conveyor_training") / RUN_NAME
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=False)
shutil.copy2(BEST_PT, DRIVE_OUTPUT / "best.pt")
results_csv = RUN_DIR / "results.csv"
if results_csv.is_file():
    shutil.copy2(results_csv, DRIVE_OUTPUT / "results.csv")
print(f"Saved model and training metadata to: {DRIVE_OUTPUT}")
files.download(str(DRIVE_OUTPUT / "best.pt"))